In [6]:
import pandas as pd
import pickle
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, average_precision_score,
    precision_recall_curve)
import time

# ── Config ────────────────────────────────────────────
MODEL_NAME    = "LogisticRegression-FraudDetection"
VERSION       = "v2.0"
RUN_NAME      = f"{MODEL_NAME}-{VERSION}"
EXPERIMENT    = f"fraud-{MODEL_NAME}"
DATASET_PATH  = "data/transactions_bancaires.csv"
OUTPUT_PATH   = f"logistic_regression_v2.pkl"

# ── Load dataset ──────────────────────────────────────
df = pd.read_csv(DATASET_PATH)
feature_cols = [
    "heure", "jour_semaine", "est_weekend", "montant_mad",
    "type_transaction", "pays_transaction", "est_etranger",
    "tx_lat", "tx_lon", "delta_km", "delta_min_last_tx",
    "nb_tx_1h", "device_type", "est_nouveau_device",
    "age_client", "segment_revenu", "type_carte"
]
X = df[feature_cols].copy()
y = df["fraude"]
for col in X.select_dtypes(include="object").columns:
    X[col] = pd.Categorical(X[col]).codes
X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Fraud rate: {y_test.mean():.4f}")

# ── Train Pipeline (scaler + model) ──────────────────
t0 = time.time()
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  LogisticRegression(
        C=0.1, class_weight="balanced",
        max_iter=1000, random_state=42))
])
pipeline.fit(X_train, y_train)
train_time = round(time.time() - t0, 2)
print(f"✅ Trained in {train_time}s")

# ── Evaluate ──────────────────────────────────────────
y_proba = pipeline.predict_proba(X_test)[:, 1]

# Optimal threshold
p, r, thresholds = precision_recall_curve(y_test, y_proba)
f1_arr  = 2*p*r/(p+r+1e-8)
best_t  = thresholds[f1_arr[:-1].argmax()]
y_pred  = (y_proba >= best_t).astype(int)

auc_roc   = round(float(roc_auc_score(y_test, y_proba)), 4)
auc_pr    = round(float(average_precision_score(y_test, y_proba)), 4)
f1        = round(float(f1_score(y_test, y_pred)), 4)
precision = round(float(precision_score(y_test, y_pred, zero_division=0)), 4)
recall    = round(float(recall_score(y_test, y_pred, zero_division=0)), 4)

print(f"\n📊 Metrics:")
print(f"  AUC-ROC   = {auc_roc}")
print(f"  AUC-PR    = {auc_pr}")
print(f"  F1        = {f1}")
print(f"  Precision = {precision}")
print(f"  Recall    = {recall}")
print(f"  Threshold = {best_t:.4f}")

# ── Save pkl ──────────────────────────────────────────
with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(pipeline, f)
print(f"\n✅ Model saved: {OUTPUT_PATH}")

# ── Log in MLflow ─────────────────────────────────────
import hashlib
with open(OUTPUT_PATH, "rb") as f:
    model_hash = "sha256:" + hashlib.sha256(f.read()).hexdigest()

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_metric("auc_roc",    auc_roc)
    mlflow.log_metric("auc_pr",     auc_pr)
    mlflow.log_metric("f1",         f1)
    mlflow.log_metric("precision",  precision)
    mlflow.log_metric("recall",     recall)
    mlflow.log_metric("n_train",    len(X_train))
    mlflow.log_metric("n_test",     len(X_test))
    mlflow.log_metric("train_time_s", train_time)
    mlflow.log_param("model_type",  "LogisticRegression+Scaler")
    mlflow.log_param("version",     VERSION)
    mlflow.log_param("model_hash_sha256", model_hash)
    mlflow.log_param("dataset",     "DS-transactions_bancaires-v2")
    mlflow.log_param("threshold",   round(float(best_t), 4))
    try:
        mlflow.sklearn.log_model(pipeline, "model",
            registered_model_name=f"FraudDetection-{MODEL_NAME}")
    except:
        pass
    run_id = run.info.run_id
    print(f"\n✅ MLflow logged: {run_id}")
    print(f"   run_name: {RUN_NAME}")
    print(f"   experiment: {EXPERIMENT}")

print("\n🎉 Pipeline complet !")
print(f"   → Upload '{OUTPUT_PATH}' dans le dashboard")
print(f"   → Model Name: '{MODEL_NAME}', Version: '{VERSION}'")
print(f"   → run_name dans MLflow: '{RUN_NAME}'")

C:\Users\dell\AppData\Local\Temp\ipykernel_24180\1138436122.py:34: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include="object").columns:


Train: 40000, Test: 10000
Fraud rate: 0.0248
✅ Trained in 0.23s

📊 Metrics:
  AUC-ROC   = 0.9318
  AUC-PR    = 0.8818
  F1        = 0.9247
  Precision = 0.9908
  Recall    = 0.8669
  Threshold = 0.7380

✅ Model saved: logistic_regression_v2.pkl


MlflowException: API request to http://mlflow:5000/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='mlflow', port=5000): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=fraud-LogisticRegression-FraudDetection (Caused by NameResolutionError("HTTPConnection(host='mlflow', port=5000): Failed to resolve 'mlflow' ([Errno 11001] getaddrinfo failed)"))

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")  # ton MLflow
mlflow.set_experiment("fraud-LogisticRegression-FraudDetection")

with mlflow.start_run(run_name="LogisticRegression-FraudDetection-v2.0") as run:
    mlflow.log_metric("auc_roc", auc_roc)
    mlflow.log_metric("f1", f1)
    # ...
